In [1]:
from datasets import load_dataset
import json, os, tempfile, random

random.seed(42)   # reproducible caption-instruction sampling

SYSTEM_PROMPT_VQA = (
    "You are a medical imaging expert. Answer concisely and base every "
    "statement on what is visible in the image."
)

SYSTEM_PROMPT_CAPTION = (
    "You are a medical imaging expert. Describe what is visible in the "
    "image concisely; do not speculate beyond it."
)

def atomic_write_json(obj, path):
    """Write JSON to `path` atomically and verify it parses back."""
    directory = os.path.dirname(os.path.abspath(path)) or "."
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", suffix=".json", dir=directory)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)              # atomic on POSIX
    except Exception:
        if os.path.exists(tmp):
            os.unlink(tmp)
        raise
    with open(path, "r", encoding="utf-8") as f:
        json.load(f)

dataset = load_dataset("MohamedAhmedAE/medical-vqa-8-datasets")

for split_name in dataset.keys():
    data_split = dataset[split_name]
    image_root = f"images/{split_name}"
    os.makedirs(image_root, exist_ok=True)

    output_data = []
    skipped_no_image  = 0
    skipped_no_answer = 0

    for idx, sample in enumerate(data_split):
        image_id = str(idx).zfill(9)
        image_filename = f"{image_id}.jpg"
        image_path = os.path.join(image_root, image_filename)

        if sample.get("image") is None:
            skipped_no_image += 1
            continue
        sample["image"].save(image_path)

        caption    = (sample.get("caption")  or "").strip()
        question   = (sample.get("question") or "").strip()
        answer_raw = (sample.get("answer")   or "").strip()
        is_caption_flag = bool(sample.get("is_caption", False))

        is_caption_effective = is_caption_flag or (
            not question and (answer_raw or caption)
        )

        if is_caption_effective:
            system_prompt = SYSTEM_PROMPT_CAPTION
            question_text = ""            # image-only: no question; the system prompt signals "describe"
            answer = answer_raw or caption
        else:
            system_prompt = SYSTEM_PROMPT_VQA
            # NOTE: prepending the caption to a VQA question gives context at TRAIN time that
            # will not exist at inference (a train/test mismatch, and a possible answer leak).
            # Set USE_CAPTION_CONTEXT = False to train VQA on the bare question only.
            USE_CAPTION_CONTEXT = False
            question_text = (
                f"Image caption: {caption}\n\n{question}"
                if (caption and USE_CAPTION_CONTEXT) else question
            )
            answer = answer_raw

        if not answer:
            skipped_no_answer += 1
            continue

        item = {
            "id": image_id,
            "image": f"{split_name}/{image_filename}",
            "data_source": sample.get("data_source", "") or "",
            "split": sample.get("split", split_name) or split_name,
            "task": "caption" if is_caption_effective else "vqa",
            "conversations": [
                {"from": "system", "value": f"{system_prompt}\n\n"},
                {"from": "human",  "value": (f"<image>\n{question_text}\n\n" if question_text else "<image>\n\n")},
                {"from": "gpt",    "value": answer},
            ],
        }
        assert all(turn.get("value") for turn in item["conversations"]), \
            f"empty turn in record {image_id}"
        output_data.append(item)

    output_file = f"medical_vqa_llava_format_{split_name}.json"
    atomic_write_json(output_data, output_file)
    print(
        f"[{split_name}] wrote {len(output_data)} samples -> {output_file} "
        f"(skipped {skipped_no_image} no-image, {skipped_no_answer} no-answer)"
    )

# sanity: caption user turns should be image-only (no question text)
cap_turns = [t["value"] for d in output_data if d["task"] == "caption"
             for t in d["conversations"] if t["from"] == "human"]
if cap_turns:
    print("example caption user turn:", repr(cap_turns[0]))
    print("all caption user turns are image-only:",
          all(t.strip() == "<image>" for t in cap_turns))
print("\nAll splits converted and verified.")

README.md:   0%|          | 0.00/749 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/82 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/82 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

...

Generating test split:   0%|          | 0/129441 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/79 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[train] wrote 667042 samples -> medical_vqa_llava_format_train.json (skipped 0 no-image, 0 no-answer)
[validation] wrote 28355 samples -> medical_vqa_llava_format_validation.json (skipped 0 no-image, 0 no-answer)
[test] wrote 129441 samples -> medical_vqa_llava_format_test.json (skipped 0 no-image, 0 no-answer)
example caption user turn: '<image>\n\n'
all caption user turns are image-only: True

All splits converted and verified.


In [2]:
# !mkdir /content/kaggle_upload

# !mv /content/medical_vqa_llava_format_test.json /content/kaggle_upload
# !mv /content/medical_vqa_llava_format_train.json /content/kaggle_upload
# !mv /content/medical_vqa_llava_format_validation.json /content/kaggle_upload

# !mv /content/images /content/kaggle_upload


In [4]:
# !pip install -U kagglehub
# !pip install -U kaggle

import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_0d24a423c0b9a810a416afc9d5200553'

# import kagglehub

# kagglehub.login()

In [5]:
import os
import json
import re
import stat
import subprocess
import sys
from pathlib import Path

# =====================================================
# Kaggle Credentials
# =====================================================
# CLI 2.x accepts two credential types, both from
# https://www.kaggle.com/settings/api :
#
#   "Generate New Token"      -> KGAT_...  KEEP THE PREFIX, it is part of
#                                the token. Goes in KAGGLE_API_TOKEN.
#   "Create Legacy API Key"   -> 32-char hex key + username, goes in
#                                kaggle.json. Works on 1.x and 2.x.
#
# Set either KAGGLE_API_TOKEN or KAGGLE_KEY in the env / Colab Secrets.
# The block below works out which one you gave it.
KAGGLE_USERNAME = "mohamedahmedae"
KAGGLE_KEY = os.environ.get("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_KEY")

if not KAGGLE_KEY:
    raise RuntimeError(
        "No credential in the environment. Set one of KAGGLE_API_TOKEN "
        "or KAGGLE_KEY, e.g.\n"
        "    from google.colab import userdata\n"
        "    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')"
    )

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)

if KAGGLE_KEY.startswith("KGAT_"):
    # --- CLI 2.x token auth ---
    token_file = kaggle_dir / "access_token"
    token_file.write_text(KAGGLE_KEY)
    os.chmod(token_file, stat.S_IRUSR | stat.S_IWUSR)  # 0600
    os.environ["KAGGLE_API_TOKEN"] = KAGGLE_KEY
    # authenticate() tries the legacy path FIRST, so a leftover kaggle.json
    # shadows a perfectly good token. Move it out of the way.
    legacy = kaggle_dir / "kaggle.json"
    if legacy.exists():
        legacy.rename(kaggle_dir / "kaggle.json.disabled")
        print("Moved a stale kaggle.json aside")
    print("Auth mode: API token (CLI 2.x)")
else:
    # --- legacy kaggle.json auth ---
    kaggle_json = kaggle_dir / "kaggle.json"
    kaggle_json.write_text(json.dumps({
        "username": KAGGLE_USERNAME,
        "key": KAGGLE_KEY,
    }))
    os.chmod(kaggle_json, stat.S_IRUSR | stat.S_IWUSR)  # 0600
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY
    print("Auth mode: legacy API key")
    if not re.fullmatch(r"[0-9a-f]{32}", KAGGLE_KEY):
        print(
            f"⚠️  Legacy keys are 32 hex chars; yours is {len(KAGGLE_KEY)}. "
            "If auth fails below, you probably have a KGAT_ token with the "
            "prefix stripped — paste the whole token instead."
        )

# =====================================================
# Dataset Configuration
# =====================================================
DATASET_SLUG = "medical-vqa-8-datasets"
UPLOAD_DIR = Path("kaggle_upload")

DATASET_METADATA = {
    "title": "Medical VQA - 8 Datasets (LLaVA format)",
    "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
}

VERSION_NOTES = "Updated via HuggingFace pipeline"

# =====================================================
# Validate Folder
# =====================================================
if not UPLOAD_DIR.exists():
    raise FileNotFoundError(
        f"Upload directory not found: {UPLOAD_DIR.resolve()}"
    )

# =====================================================
# Auth Sanity Check (cheap call, fails fast on bad creds)
# =====================================================
print("\nVerifying Kaggle credentials...")
auth_check = subprocess.run(
    ["kaggle", "datasets", "list", "-m"],
    capture_output=True,
    text=True,
)
auth_out = (auth_check.stdout or "") + (auth_check.stderr or "")

# CLI 2.x prints the auth banner and STILL exits 0, so the return code alone
# is not enough — this is why the old check reported "Credentials OK" on
# credentials that then failed at the first uploaded file.
if auth_check.returncode != 0 or re.search(
    r"authentication required|sign up for a kaggle account|kaggle auth login",
    auth_out,
    re.I,
):
    print(auth_out)
    raise RuntimeError(
        "Kaggle auth failed. At https://www.kaggle.com/settings/api either:\n"
        "  (a) Generate New Token -> put the full KGAT_... value, prefix "
        "included, in KAGGLE_API_TOKEN; or\n"
        "  (b) Legacy API Credentials -> Create Legacy API Key -> put the "
        "32-char hex key in KAGGLE_KEY."
    )
print("✓ Credentials OK\n")

# =====================================================
# Create Metadata File
# =====================================================
metadata_file = UPLOAD_DIR / "dataset-metadata.json"
with open(metadata_file, "w") as f:
    json.dump(DATASET_METADATA, f, indent=2)
print(f"✓ Metadata saved to {metadata_file}")

# =====================================================
# Folder Statistics
# =====================================================
total_size = sum(
    f.stat().st_size for f in UPLOAD_DIR.rglob("*") if f.is_file()
)
num_files = sum(1 for f in UPLOAD_DIR.rglob("*") if f.is_file())
print(f"Folder size: {total_size / (1024**3):.2f} GB")
print(f"Number of files: {num_files}")

# With --dir-mode zip the file count barely matters: each top-level directory
# goes up as ONE archive, which Kaggle unpacks server-side, so the published
# dataset still lists files individually.
print(f"Blobs to upload: {len(list(UPLOAD_DIR.iterdir())) - 1}")

# =====================================================
# Check if Dataset Exists
# =====================================================
print("\nChecking if dataset exists...")
check_result = subprocess.run(
    ["kaggle", "datasets", "status", f"{KAGGLE_USERNAME}/{DATASET_SLUG}"],
    capture_output=True,
    text=True,
)
check_out = (check_result.stdout or "") + (check_result.stderr or "")
dataset_exists = check_result.returncode == 0 and not re.search(
    r"404|not found", check_out, re.I
)
print(f"Dataset exists: {dataset_exists}")

# =====================================================
# Create or Update
# =====================================================
# --keep-tabular stops Kaggle rewriting the LLaVA JSONs as CSV, which would
# flatten the nested "conversations" structure.
if not dataset_exists:
    print("\nCreating new dataset...")
    cmd = [
        "kaggle", "datasets", "create",
        "--path", str(UPLOAD_DIR),
        "--dir-mode", "zip",
        "--keep-tabular",
    ]
else:
    print("\nUpdating existing dataset...")
    cmd = [
        "kaggle", "datasets", "version",
        "--path", str(UPLOAD_DIR),
        "--message", VERSION_NOTES,
        "--dir-mode", "zip",
        "--keep-tabular",
    ]

# In a notebook, a bare subprocess.run(cmd) sends the child's output to the
# KERNEL's file descriptors, which the cell never renders — that is why the
# original run showed "Return code: 1" and nothing else. Pipe it back in.
# Raw os.read rather than line iteration, because the progress bars use
# carriage returns and line-buffered reads stall on them.
print(f"Running: {' '.join(cmd)}\n")
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
while True:
    chunk = os.read(proc.stdout.fileno(), 4096)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.stdout.close()
returncode = proc.wait()

# =====================================================
# Output Results
# =====================================================
print(f"\nReturn code: {returncode}")
if returncode == 0:
    print("\n✅ Upload completed successfully!")
    print(f"https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")
else:
    print("\n❌ Upload failed. The CLI's actual error is in the block above.")

Moved a stale kaggle.json aside
Auth mode: API token (CLI 2.x)

Verifying Kaggle credentials...
✓ Credentials OK

✓ Metadata saved to kaggle_upload/dataset-metadata.json
Folder size: 26.69 GB
Number of files: 824842
Blobs to upload: 4

Checking if dataset exists...
Dataset exists: False

Creating new dataset...
Running: kaggle datasets create --path kaggle_upload --dir-mode zip --keep-tabular

Starting upload for file medical_vqa_llava_format_validation.json
100%|██████████| 19.1M/19.1M [00:01<00:00, 11.2MB/s]
Upload successful: medical_vqa_llava_format_validation.json (19MB)
Starting upload for file images.zip
100%|██████████| 25.3G/25.3G [10:49<00:00, 41.8MB/s]
Upload successful: images.zip (25GB)
Starting upload for file medical_vqa_llava_format_train.json
100%|██████████| 428M/428M [00:11<00:00, 40.3MB/s]
Upload successful: medical_vqa_llava_format_train.json (428MB)
Starting upload for file medical_vqa_llava_format_test.json
100%|██████████| 83.2M/83.2M [00:03<00:00, 28.3MB/s]
Upl